In [ ]:
# 1. Désinstallation propre de peft, transformers, torch, torchvision, torchaudio
!pip uninstall -y peft transformers torch torchvision torchaudio

# 2. Réinstallation compatible de torch + torchvision + torchaudio (GPU CUDA 11.8)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# 3. Réinstallation de transformers
!pip install -U transformers

# 4. Installation de la dernière version dev de peft depuis GitHub
!pip install git+https://github.com/huggingface/peft.git

# 5. Redémarrage automatique du runtime pour prendre en compte les changements
import os
os.kill(os.getpid(), 9)


Found existing installation: peft 0.16.1.dev0
Uninstalling peft-0.16.1.dev0:
  Successfully uninstalled peft-0.16.1.dev0
Found existing installation: transformers 4.53.1
Uninstalling transformers-4.53.1:
  Successfully uninstalled transformers-4.53.1
Found existing installation: torch 2.7.1+cu118
Uninstalling torch-2.7.1+cu118:
  Successfully uninstalled torch-2.7.1+cu118
Found existing installation: torchvision 0.22.1+cu118
Uninstalling torchvision-0.22.1+cu118:
  Successfully uninstalled torchvision-0.22.1+cu118
Found existing installation: torchaudio 2.7.1+cu118
Uninstalling torchaudio-2.7.1+cu118:
  Successfully uninstalled torchaudio-2.7.1+cu118
Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (28 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (6.1 kB)
  Using cached h

In [ ]:
import peft
print("Version de peft :", peft.__version__)

from peft import merge_and_unload
print("merge_and_unload est importé avec succès !")


Version de peft : 0.16.1.dev0


ImportError: cannot import name 'merge_and_unload' from 'peft' (/usr/local/lib/python3.11/dist-packages/peft/__init__.py)

In [ ]:
import peft
print(dir(peft))


['AdaLoraConfig', 'AdaLoraModel', 'AdaptionPromptConfig', 'AdaptionPromptModel', 'AutoPeftModel', 'AutoPeftModelForCausalLM', 'AutoPeftModelForFeatureExtraction', 'AutoPeftModelForQuestionAnswering', 'AutoPeftModelForSeq2SeqLM', 'AutoPeftModelForSequenceClassification', 'AutoPeftModelForTokenClassification', 'BOFTConfig', 'BOFTModel', 'BoneConfig', 'BoneModel', 'C3AConfig', 'C3AModel', 'CPTConfig', 'CPTEmbedding', 'EvaConfig', 'FourierFTConfig', 'FourierFTModel', 'HRAConfig', 'HRAModel', 'IA3Config', 'IA3Model', 'LNTuningConfig', 'LNTuningModel', 'LoHaConfig', 'LoHaModel', 'LoKrConfig', 'LoKrModel', 'LoftQConfig', 'LoraConfig', 'LoraModel', 'LoraRuntimeConfig', 'MODEL_TYPE_TO_PEFT_MODEL_MAPPING', 'MultitaskPromptTuningConfig', 'MultitaskPromptTuningInit', 'OFTConfig', 'OFTModel', 'PEFT_TYPE_TO_CONFIG_MAPPING', 'PEFT_TYPE_TO_MIXED_MODEL_MAPPING', 'PEFT_TYPE_TO_TUNER_MAPPING', 'PeftConfig', 'PeftMixedModel', 'PeftModel', 'PeftModelForCausalLM', 'PeftModelForFeatureExtraction', 'PeftModel

In [ ]:
import torch
from peft import PeftModel

def merge_and_unload(peft_model: PeftModel) -> torch.nn.Module:
    """
    Merge LoRA adapter weights into the base model and unload PEFT adapters.

    Args:
        peft_model (`PeftModel`): The PEFT model with LoRA adapters.

    Returns:
        `torch.nn.Module`: The base model with merged weights (without adapters).
    """
    # Check that the model is a PeftModel
    if not isinstance(peft_model, PeftModel):
        raise ValueError("The model passed is not a PeftModel instance.")

    base_model = peft_model.base_model

    # Iterate over all modules and merge LoRA weights
    for name, module in base_model.named_modules():
        if hasattr(module, "merge_weights"):
            module.merge_weights()

    # Remove PEFT adapters to free memory
    peft_model.active_adapter = None

    return base_model



In [ ]:

import os,torch, wandb

In [ ]:
base_model="mistralai/Mistral-7B-v0.1"

In [ ]:
from huggingface_hub import login
login(new_session=False)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model_reload = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    device_map="auto",
    offload_folder="offload",
    torch_dtype=torch.float16
)





Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
lora_model = PeftModel.from_pretrained(base_model_reload, "rouabenyahia/FineTuningModel")

In [ ]:
from peft import PeftModel, PeftConfig
merged_model = merge_and_unload(lora_model)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
merged_model.push_to_hub("rouabenyahia/FineTuningModel")
tokenizer.push_to_hub("rouabenyahia/FineTuningModel")

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/rouabenyahia/FineTuningModel/commit/90128db5aff86a367fb957b5aef0dc7620fc4bf2', commit_message='Upload tokenizer', commit_description='', oid='90128db5aff86a367fb957b5aef0dc7620fc4bf2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rouabenyahia/FineTuningModel', endpoint='https://huggingface.co', repo_type='model', repo_id='rouabenyahia/FineTuningModel'), pr_revision=None, pr_num=None)